In [ ]:
# Copyright (C) 2026 Advanced Micro Devices, Inc. All rights reserved.
#
# SPDX-License-Identifier: BSD-3-Clause
#
# Portions of this file consist of AI-generated content. AI-assisted
# content has been reviewed and validated by the authors.

# Voice and MoveIt on the digital twin

This lesson is a vision-language-action loop on the twin. You speak. Gemma 4 E4B-IT,
a multimodal language model, takes that voice audio directly and turns it into one command. MoveIt plans the arm motion and the
ros2_control trajectory controller plays it. YOLO-pose and MediaPipe Hands
read your body and your hand when the command is "copy my movements".

The twin is useful because the same client drives it and the real arm:

1. **It moves like the arm** — MoveIt and the same ros2_control controllers, inside the same joint speed limits.
2. **It sees like the arm** — a camera rendered where the real one is mounted.
3. **It feels like the arm** — the gripper action reports a stall when the jaws meet something.

With those three checks done, section 4 is the console: you speak, Gemma 4 E4B-IT
picks a behavior, and MoveIt carries it out on the twin.

What is simulated: the arm, the wrist camera, and the table (MuJoCo). What is
real: your webcam and microphone, Gemma 4 E4B-IT on the iGPU, Silero VAD on the CPU,
YOLOv26s-pose on the NPU, MediaPipe Hands on the CPU, and the behaviors.
Sections 0 to 3 check the twin. Section 0 starts it on its own ROS domain,
and this kernel drives it with `MoveItArm`, the same client the pipeline uses
for the real arm. Section 4 is where you give commands.


## Compute placement

Each part of the loop runs on the processor that fits it. The lesson is the voice command and the MoveIt motion, plus pose and hands for mimic.

`./run_pipeline.sh` also pins three programs to one CPU core each, from `cores` in `config/pipeline.yaml`. This notebook starts the same programs and passes no core, so each one may use any core. The kernel holds the arm client, the behaviors, and the console. Perception, `llama-server`, the mount camera, and the MuJoCo bringup are separate processes. `run_notebooks.sh` starts `llama-server` before the kernel, and that server keeps every core.

<table style="border-collapse: collapse">
<thead>
<tr><th style="background-color: rgba(128, 128, 128, 0.2); border: 1px solid rgba(128, 128, 128, 0.4)">Task</th><th style="border: 1px solid rgba(128, 128, 128, 0.4)">Model or algorithm</th><th style="border: 1px solid rgba(128, 128, 128, 0.4)">What it does</th><th style="border: 1px solid rgba(128, 128, 128, 0.4)">Runs on</th><th style="border: 1px solid rgba(128, 128, 128, 0.4)">Runtime</th></tr>
</thead>
<tbody>
<tr style="background-color: rgba(0, 0, 0, 0)"><td rowspan="2" valign="top" style="background-color: rgba(128, 128, 128, 0.2); vertical-align: top; border: 1px solid rgba(128, 128, 128, 0.4)">Voice control</td><td style="border: 1px solid rgba(128, 128, 128, 0.4)">Silero VAD</td><td style="border: 1px solid rgba(128, 128, 128, 0.4)">marks 20 ms frames that contain speech</td><td style="border: 1px solid rgba(128, 128, 128, 0.4)">CPU (core 4)</td><td style="border: 1px solid rgba(128, 128, 128, 0.4)">ONNX Runtime</td></tr>
<tr style="background-color: rgba(0, 0, 0, 0)"><td style="border: 1px solid rgba(128, 128, 128, 0.4)">Gemma 4 E4B-IT</td><td style="border: 1px solid rgba(128, 128, 128, 0.4)">multimodal language model; voice audio in, one command out</td><td style="border: 1px solid rgba(128, 128, 128, 0.4)">iGPU</td><td style="border: 1px solid rgba(128, 128, 128, 0.4)"><code>llama-server</code> (HIP)</td></tr>
<tr style="background-color: rgba(0, 0, 0, 0)"><td rowspan="2" valign="top" style="background-color: rgba(128, 128, 128, 0.2); vertical-align: top; border: 1px solid rgba(128, 128, 128, 0.4)">Your movement</td><td style="border: 1px solid rgba(128, 128, 128, 0.4)">YOLOv26s-pose</td><td style="border: 1px solid rgba(128, 128, 128, 0.4)">finds your body and wrist</td><td style="border: 1px solid rgba(128, 128, 128, 0.4)">NPU</td><td style="border: 1px solid rgba(128, 128, 128, 0.4)">ONNX Runtime, VitisAI EP</td></tr>
<tr style="background-color: rgba(0, 0, 0, 0)"><td style="border: 1px solid rgba(128, 128, 128, 0.4)">MediaPipe Hands</td><td style="border: 1px solid rgba(128, 128, 128, 0.4)">reads the pinch and the roll</td><td style="border: 1px solid rgba(128, 128, 128, 0.4)">CPU (core 4)</td><td style="border: 1px solid rgba(128, 128, 128, 0.4)">MediaPipe Tasks</td></tr>
<tr style="background-color: rgba(0, 0, 0, 0)"><td style="background-color: rgba(128, 128, 128, 0.2); vertical-align: top; border: 1px solid rgba(128, 128, 128, 0.4)">The object</td><td style="border: 1px solid rgba(128, 128, 128, 0.4)">YOLOv26s-detect</td><td style="border: 1px solid rgba(128, 128, 128, 0.4)">finds the named object in the wrist camera</td><td style="border: 1px solid rgba(128, 128, 128, 0.4)">NPU</td><td style="border: 1px solid rgba(128, 128, 128, 0.4)">ONNX Runtime, VitisAI EP</td></tr>
<tr style="background-color: rgba(0, 0, 0, 0)"><td style="background-color: rgba(128, 128, 128, 0.2); vertical-align: top; border: 1px solid rgba(128, 128, 128, 0.4)">Arm motion</td><td style="border: 1px solid rgba(128, 128, 128, 0.4)">MoveIt</td><td style="border: 1px solid rgba(128, 128, 128, 0.4)">plans the command and the gripper action reports a stall</td><td style="border: 1px solid rgba(128, 128, 128, 0.4)">CPU (core 3)</td><td style="border: 1px solid rgba(128, 128, 128, 0.4)">MoveIt Task Constructor, ros2_control</td></tr>
<tr style="background-color: rgba(0, 0, 0, 0)"><td style="background-color: rgba(128, 128, 128, 0.2); vertical-align: top; border: 1px solid rgba(128, 128, 128, 0.4)">Digital twin</td><td style="border: 1px solid rgba(128, 128, 128, 0.4)">MuJoCo</td><td style="border: 1px solid rgba(128, 128, 128, 0.4)">simulates the arm and the table</td><td style="border: 1px solid rgba(128, 128, 128, 0.4)">CPU (core 2), rendering on the iGPU</td><td style="border: 1px solid rgba(128, 128, 128, 0.4)">MuJoCo, EGL</td></tr>
</tbody>
</table>

Where each program sits. "Any" means the scheduler may use every core.

<table style="border-collapse: collapse">
<thead>
<tr><th style="background-color: rgba(128, 128, 128, 0.2); border: 1px solid rgba(128, 128, 128, 0.4)">Program</th><th style="border: 1px solid rgba(128, 128, 128, 0.4)">What it runs</th><th style="border: 1px solid rgba(128, 128, 128, 0.4)">This notebook</th><th style="border: 1px solid rgba(128, 128, 128, 0.4)"><code>./run_pipeline.sh</code></th></tr>
</thead>
<tbody>
<tr style="background-color: rgba(0, 0, 0, 0)"><td style="border: 1px solid rgba(128, 128, 128, 0.4)">Perception</td><td style="border: 1px solid rgba(128, 128, 128, 0.4)">YOLO pose, YOLO detect, MediaPipe, Silero</td><td style="border: 1px solid rgba(128, 128, 128, 0.4)">its own process, any core</td><td style="border: 1px solid rgba(128, 128, 128, 0.4)">core 4</td></tr>
<tr style="background-color: rgba(0, 0, 0, 0)"><td style="border: 1px solid rgba(128, 128, 128, 0.4)"><code>llama-server</code></td><td style="border: 1px solid rgba(128, 128, 128, 0.4)">Gemma 4 on the iGPU</td><td style="border: 1px solid rgba(128, 128, 128, 0.4)">its own process, any core</td><td style="border: 1px solid rgba(128, 128, 128, 0.4)">core 4 when perception starts it</td></tr>
<tr style="background-color: rgba(0, 0, 0, 0)"><td style="border: 1px solid rgba(128, 128, 128, 0.4)">Mount webcam</td><td style="border: 1px solid rgba(128, 128, 128, 0.4)"><code>v4l2_camera</code></td><td style="border: 1px solid rgba(128, 128, 128, 0.4)">its own process, any core</td><td style="border: 1px solid rgba(128, 128, 128, 0.4)">core 4</td></tr>
<tr style="background-color: rgba(0, 0, 0, 0)"><td style="border: 1px solid rgba(128, 128, 128, 0.4)">MuJoCo bringup</td><td style="border: 1px solid rgba(128, 128, 128, 0.4)">the twin, the wrist camera, the scene camera</td><td style="border: 1px solid rgba(128, 128, 128, 0.4)">its own process, any core</td><td style="border: 1px solid rgba(128, 128, 128, 0.4)">core 2</td></tr>
<tr style="background-color: rgba(0, 0, 0, 0)"><td style="border: 1px solid rgba(128, 128, 128, 0.4)"><code>move_group</code> and Servo</td><td style="border: 1px solid rgba(128, 128, 128, 0.4)">planning, and the streamed joints</td><td style="border: 1px solid rgba(128, 128, 128, 0.4)">the MuJoCo launch, any core</td><td style="border: 1px solid rgba(128, 128, 128, 0.4)">core 3</td></tr>
<tr style="background-color: rgba(0, 0, 0, 0)"><td style="border: 1px solid rgba(128, 128, 128, 0.4)">Control</td><td style="border: 1px solid rgba(128, 128, 128, 0.4)">the arm client and the behaviors</td><td style="border: 1px solid rgba(128, 128, 128, 0.4)">this kernel, any core</td><td style="border: 1px solid rgba(128, 128, 128, 0.4)">core 3, its own process</td></tr>
<tr style="background-color: rgba(0, 0, 0, 0)"><td style="border: 1px solid rgba(128, 128, 128, 0.4)">Console</td><td style="border: 1px solid rgba(128, 128, 128, 0.4)">the window</td><td style="border: 1px solid rgba(128, 128, 128, 0.4)">this kernel, any core</td><td style="border: 1px solid rgba(128, 128, 128, 0.4)">core 2, its own process</td></tr>
</tbody>
</table>


In [ ]:
# setup: section 4 uses the pipeline's split. Perception runs the models and
# the microphone. This kernel keeps the arm and runs the behaviors. The console
# only draws what those two publish.
import importlib.util
import threading, time

# hide harmless warnings from other libraries
from vvla_workshop.quiet import hush_benign_warnings

hush_benign_warnings()  # silence known-benign third-party log noise

# Keep logger.info off the cell. Jupyter prints stderr in red, and that
# stream is where an info handler lands.
import logging
logging.getLogger().setLevel(logging.WARNING)

# images (OpenCV), arrays (NumPy), plots, and the buttons and text box used in section 4
import cv2, numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display
from vvla_workshop.ui_poll import ui_events  # jupyter_ui_poll's, with the stop button working on cells run meanwhile

# read the pipeline settings, then switch them from the real arm to the twin
from vla_pipeline.utils.config import load_config

cfg = load_config()
cfg["robot"]["backend"] = "sim"  # the MuJoCo twin instead of the Feetech bus


## 0. A body

The twin is the MuJoCo scene under ros2_control and MoveIt. The next cell starts that bringup and connects this kernel to it with `MoveItArm`. Joint positions from here on are radians under the URDF joint names.


In [ ]:
# the MuJoCo MoveIt bringup, and this kernel's arm
from vvla_workshop.quiet import hush_benign_warnings

hush_benign_warnings()  # MoveIt INFO goes to stderr; Jupyter would show it in red
from vvla_workshop.twin_ros import TwinSession

twin = TwinSession(cfg)
arm, wrist, view = twin.arm, twin.armcam, twin.view
model = arm.robot.get_robot_model()
print(f"{model.name}: {len(arm.arm_joints)} arm joints, gripper {arm.gripper_joint}")
print(f"twin pid {twin.pid} on ROS domain {twin.domain_id}")
print(f"sim clock {arm.state_time():.3f} s")


## 1. It moves like the arm

`go_to` with a joint goal asks MoveIt for the move. MoveIt plans it and times it within each joint's `max_velocity` and `max_acceleration` in `joint_limits.yaml`, and the joint trajectory controller plays it on the twin, as it does on the arm. The samples below are the twin's joint states during a 60 degree move of the base. `go_to_rest` returns to the SRDF home pose and closes the jaws.


In [ ]:
# one planned move on the base, in radians, then back to the SRDF home pose
from sensor_msgs.msg import JointState
from vla_pipeline.robot.moveit_arm import moveit_configs

joint = "shoulder_pan_joint"
delta = 1.047  # 60 degrees
limit = moveit_configs().joint_limits["robot_description_planning"]["joint_limits"][joint]["max_velocity"]
q0 = arm.get_joints()[joint]

# record the twin's joint states, stamped on its clock, while MoveIt plans and runs the move
samples = []
sub = arm.node.create_subscription(
    JointState, "/joint_states",
    lambda m: samples.append((m.header.stamp.sec + m.header.stamp.nanosec * 1e-9, dict(zip(m.name, m.position))[joint])), 100,
)
print("move:", arm.go_to({joint: q0 + delta}))
arm.node.destroy_subscription(sub)

t, q = np.array(samples).T
t, q = t - t[0], q - q0
v = np.gradient(q, t)
print(f"peak speed {v.max():.2f} rad/s, max_velocity {limit:.2f} rad/s in joint_limits.yaml")

fig, ax = plt.subplots(figsize=(8, 3))
ax.axhline(delta, color="k", ls="--", lw=1, label="goal")
ax.plot(t, q, label="position (rad)")
ax.plot(t, v, label="speed (rad/s)")
ax.axhline(limit, color="r", ls=":", lw=1, label="max_velocity")
ax.set_xlabel("seconds on the twin's clock")
ax.legend()
plt.show()
arm.go_to_rest()


## 2. It sees like the arm

The wrist camera and the scene camera are MuJoCo camera plugins. They publish ROS images. `read()` returns `(ok, bgr)`, the same shape as a webcam. The next cell moves to the SRDF ready pose with the jaws open and shows both frames.


In [ ]:
# wrist and scene cameras from the MuJoCo plugins
arm.go_to_named("ready")
arm.go_to_named("open")
ok_w, w_img = wrist.read()
ok_v, v_img = view.read()
assert ok_w and ok_v, "a camera published no frame"
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for ax, img, title in zip(axes, (w_img, v_img), ("wrist camera", "scene camera")):
    ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    ax.set_title(title)
    ax.axis("off")
plt.show()


## 3. It feels like the arm

The gripper is the parallel gripper action. An empty close is asked for the SRDF close pose. The action reports whether the jaws stalled on the way there.


In [ ]:
# empty close on the gripper action, then open and park
arm.go_to_named("ready")
close = arm.named_state("close")[arm.gripper_joint]
stalled = arm.grip(close)
print(f"empty close stalled: {stalled}")
arm.go_to_named("open")
arm.go_to_rest()


## 4. Control the digital twin

From here on you drive the twin with your voice, or by typing commands. It is the same arm that section 0 started, and it keeps running. The work is split the same way as `./run_pipeline.sh`:

1. **Perception** is its own process. Gemma 4 E4B-IT (iGPU) turns a hop of voice into one command. YOLOv26s-pose (NPU) and MediaPipe Hands (CPU) watch the mount webcam. YOLOv26s-detect (NPU) reads the wrist camera. That process publishes what it saw and what it heard.
2. **This kernel** is the control side. It keeps the `MoveItArm` from section 0 and runs the behavior for that command. It does not load the models or open the microphone.
3. **The console** only draws. It shows the overlays, the wrist view, the twin, and the gripper current that the arm publishes. It does not run a model and it does not read the arm itself.

The picture below is that console.


In [ ]:
# Start the voice console from a clean scene.
assert twin.reset_scene(), "section 4 could not reset the twin"
print("SECTION4_RESET")
# the twin from section 0 is this kernel's arm
assert importlib.util.find_spec("rclpy"), (
    "section 4 needs rclpy: source /opt/ros/jazzy/setup.bash, then restart ./run_notebooks.sh"
)
print(f"twin node pid {twin.pid} on ROS domain {twin.domain_id} (this machine only)")
print(f"arm: {type(arm).__name__}")


### Watching you

The next cell starts the mount webcam and the perception process, then shows ten seconds of the console. The tiles are the overlays perception publishes. The gripper current along the bottom is the arm's own feed, in milliamps, full scale at the servo's stall current.

In [ ]:
# Perception watches the cameras and the microphone. This kernel only drives the arm.
import os
import shutil
import signal
import subprocess
import sys
import tempfile
from pathlib import Path

from vla_pipeline.console import TwinConsole, jpeg
from vla_pipeline.llm.intents import keyword_parse
from vla_pipeline.robot.ros2_control_arm import Spinner
from vla_pipeline.user_robot_control import PipelineContext, UserRobotControl, build_registry
from vla_pipeline.utils.config import REPO_ROOT
from vla_pipeline.utils.gesture_map import mapper_from_config
from vla_pipeline.wire import ConsoleLink, ControlLink


def _side(cmd):
    """Run cmd in its own session. It dies with this kernel. Its log stays off this cell."""
    fd, name = tempfile.mkstemp(prefix="notebook_", suffix=".log")
    log = os.fdopen(fd, "w")
    proc = subprocess.Popen(
        [sys.executable, "-m", "so101_sim.parent_death", *cmd],
        stdout=log, stderr=subprocess.STDOUT,
        env=os.environ.copy(),
        start_new_session=True,
    )
    proc.log_file = log
    proc.log_path = Path(name)
    return proc


def _stop_side(proc):
    """SIGTERM to the parent_death wrapper, which stops the process it started."""
    if proc is None or proc.poll() is not None:
        return
    proc.send_signal(signal.SIGTERM)
    try:
        proc.wait(timeout=10)
    except subprocess.TimeoutExpired:
        try:
            os.killpg(proc.pid, signal.SIGKILL)
        except ProcessLookupError:
            pass
        proc.wait(timeout=5)


if "control" not in globals():
    ctx = PipelineContext(
        cfg=cfg,
        arm=arm,
        mapper=mapper_from_config(cfg),
        registry=build_registry(),
    )
    # Typed text uses the keyword stand-in. Spoken audio arrives as a command
    # the perception process already parsed.
    robot_voice = UserRobotControl(ctx, keyword_parse)
    control = ControlLink(arm.node, ctx, robot_voice, twin.reset_scene)

if "console" not in globals():
    console = TwinConsole(
        ctx, view=view, hint="say 'stop robot' to halt the arm", restore=twin.reset_scene,
    )
    console.use_tiles()
    # The arm publishes its joints. The console paints that publication and does not read the arm.
    arm.on_cached_joints = control.set_joints

    def say(message: str) -> None:
        print(message)
        control.say = message
        control.publish_state()

    ctx.say = say
    observer = Spinner("notebook_console")
    console_link = ConsoleLink(observer.node, console)

ros2 = shutil.which("ros2")
if ros2 is None:
    raise RuntimeError("ros2 is not on PATH: source /opt/ros/jazzy/setup.bash, then restart the kernel")
launch = REPO_ROOT / "launch" / "strix_vla.launch.py"
if "camera_proc" not in globals() or camera_proc.poll() is not None:
    camera_proc = _side([
        sys.executable, ros2, "launch", str(launch),
        f"config:={REPO_ROOT / 'config' / 'pipeline.yaml'}",
        "arm:=false", "cameras:=true", "digital_twin:=true",
    ])
if "perception_proc" not in globals() or perception_proc.poll() is not None:
    from rclpy.qos import DurabilityPolicy, HistoryPolicy, QoSProfile, ReliabilityPolicy
    from std_msgs.msg import String

    from vla_pipeline.ipc import STATUS

    # One status message means the models are loaded. The process exiting means they are not.
    ready = threading.Event()
    status_qos = QoSProfile(
        reliability=ReliabilityPolicy.RELIABLE, history=HistoryPolicy.KEEP_LAST, depth=1,
        durability=DurabilityPolicy.TRANSIENT_LOCAL,
    )
    observer.node.create_subscription(String, STATUS, lambda _msg: ready.set(), status_qos)
    perception_proc = _side([
        sys.executable, "-m", "vla_pipeline.perception", "--digital-twin",
    ])

    def _perception_ended() -> None:
        perception_proc.wait()
        ready.set()

    threading.Thread(target=_perception_ended, daemon=True, name="perception-exit").start()
    ready.wait()
    if perception_proc.poll() is not None:
        raise RuntimeError(f"perception exited (log {perception_proc.log_path})")

preview = widgets.Image(format="jpeg", layout=widgets.Layout(width="1200px"))
display(preview)
for _ in range(10 * 15):
    preview.value = jpeg(console.compose())
    time.sleep(1 / 15)
if perception_proc.poll() is not None:
    print(f"perception is not running (log {perception_proc.log_path})")


### Listening to you

Your voice reaches the robot in three steps. The perception process from the previous cell owns the first two. This kernel runs the third.

1. **Silero VAD** (CPU) marks the 20 ms frames that contain speech and are louder than −35 dBFS. A window of that audio is sent on a hop. There is no wait for silence.
2. **Gemma 4 E4B-IT** (iGPU, through `llama-server`) is a multimodal language model. It takes the voice audio directly and answers with one robot command, for example `pick_place` with the object `ball`. Perception publishes that command.
3. The **behavior** for that command runs on the arm through MoveIt. A new command stops the one that is running.

"stop robot" stops the arm before another hop is classified as a new command. "stop" alone does nothing. Noise is ignored. `handle_transcript` (TODO 3.2) makes these checks.

Which model hears you is `voice_intent` in `config/pipeline.yaml`. Speech uses the same `gemma4-e4b-it` path as `./run_pipeline.sh`. The text box uses the keyword stand-in, the same path as `./run_pipeline.sh --text-commands`. Without the Gemma model files, that stand-in is also what hears the microphone.


In [ ]:
# The perception process already owns the microphone. A typed line in the next cell
# is delivered to the same behavior queue a spoken command arrives on.
print("Speak in the console cell, or type a command there.")


### Controlling the digital twin

Run the next cell, then try two things, in this order:

1. Say something that is not about the robot, for example "what's the weather like today". Gemma finds no command, the console says "I didn't catch a command in that", and the arm does not move.
2. Say "wave at me". Gemma picks `wave`, and the arm waves.

Under the console, **stop** ends the cell and parks the arm, **reset** turns gray and says "resetting..." until the arm is parked and the objects are back (another click does nothing until then), and **end session** also ends the cell (run the cell again to resume). Saying "stop" only halts the behavior, so a later command in the same session still works. The notebook runs one cell at a time, so click **stop** or **end session** (or the kernel's stop button) before you run the clean-up cell.

| Say | What happens |
|---|---|
| "copy my movements" | The arm follows your right hand: where it is in the image, how close it is to the camera, its tilt, and your pinch for the gripper. |
| "pick up the ball" (or bottle, book, toothbrush) | MoveIt plans a pick of the object the command names, lifts it, and holds it. |
| "hold this" | The jaws close on what is between them and keep holding. If you then say "copy my movements", your pinch controls the gripper. |
| "release", "drop it" | The jaws open. |
| "place it", "set it down" | After a pickup, the arm lowers the object onto the table under the jaws, opens them, and returns home. |
| "wave", "dance" | The arm waves, or dances to a beat. |
| "go home" | The arm returns to its rest pose. |
| "stop robot" | Everything stops at once. |

All commands: [Voice commands → behaviors](../../../README.md#voice-commands--behaviors).


In [ ]:
# the console image with its buttons and text box, redrawn FPS times a second
FPS = 15
# Start with a frame, not empty: JupyterLab may not re-measure an image that grows later.
screen = widgets.Image(value=jpeg(console.compose()), format="jpeg", layout=widgets.Layout(width="1200px"))
stop = widgets.Button(description="stop", button_style="danger")
reset = widgets.Button(description="reset", button_style="warning")
typed = widgets.Text(placeholder="or type a command, then Enter", continuous_update=False,
                     layout=widgets.Layout(width="400px"))
end = widgets.Button(description="end session")
finished = threading.Event()  # set by stop or end session


def submit(change):
    """Enter in the text box sends the typed command, the same way as a spoken one."""
    text = change["new"].strip()
    if text:
        typed.value = ""
        robot_voice.on_transcript(text)


def end_session(_=None):
    """Ends the loop below and the running behavior."""
    finished.set()
    ctx.stop_event.set()
    robot_voice.stop()  # wakes the dispatch thread, which then sees finished


def on_reset(_):
    """Gray the button, redraw, then reset. A click while it is gray does nothing."""
    if reset.disabled:
        return

    def show_busy():
        reset.disabled = True
        reset.description = "resetting..."
        reset.button_style = ""
        screen.value = jpeg(console.compose())

    console.reset_scene(on_busy=show_busy)


stop.on_click(end_session)  # the red button ends the cell; saying "stop" only halts the behavior
reset.on_click(on_reset)
typed.observe(submit, names="value")
end.on_click(end_session)
display(widgets.VBox([screen, widgets.HBox([stop, reset, typed, end])]))

# run commands on their own thread, one at a time, and redraw the console until stop or end session
ctx.say("Ready. Try: 'copy my movements', 'hold this', 'pick up the ball', 'pick up the bottle', "
        "'pick up the book', 'pick up the toothbrush', 'wave', 'dance', 'go home' - or 'stop robot'.")
# a command spoken while the cameras were warming up is already queued, and runs now
worker = threading.Thread(target=robot_voice.run, args=(finished,), daemon=True, name="dispatch")
worker.start()
try:
    # running this cell again while it runs (JupyterLab's run and advance) also ends the session
    with ui_events(on_rerun=end_session) as poll:
        while not finished.is_set():
            t0 = time.perf_counter()
            poll(10)  # deliver button clicks and typed commands while this cell runs
            # Stay gray until the redraw after the reset, so a click queued during it is ignored.
            if reset.disabled and not console.resetting:
                reset.description = "reset"
                reset.button_style = "warning"
                reset.disabled = False
            screen.value = jpeg(console.compose())
            time.sleep(max(0.0, 1.0 / FPS - (time.perf_counter() - t0)))
finally:
    # stop, end session, or the kernel's stop button: stop UserRobotControl, wait for the last behavior, park the arm
    end_session()
    worker.join(timeout=10)
    arm.go_to_rest()
    print("session ended, arm parked")

### Running it outside the notebook

This section ran the pipeline itself (`vla_pipeline/`), which holds the finished code of every TODO. Your versions are in `workshop/project/`:

- `models/`: `npu.py` (TODOs 1.1-1.3), `hands.py` (1.4), `llm.py` (1.5-1.6)
- `ros/`: `camera_client.py` (2.3), `intent_node.py` (2.10-2.11)
- `integration/`: `dispatch.py` (3.1-3.2), `mimic.py` (3.3), `behaviors.py` (3.4)

`python -m vvla_workshop.selftest` checks all of them; `--reference` runs the same checks on the pipeline's code. To drive the twin from a terminal, with the console in its own desktop window, run `./run_pipeline.sh --digital-twin`. That starts the same three pieces this section did: perception, control, and the console.


## Clean up

Click **stop** or **end session** under the console (or the kernel's stop button) before you run this cell. It stops perception and the mount camera, then closes the console and the twin. The microphone closes with perception.


In [ ]:
# the console cell ended its session; this also ends a dispatch thread it left running
finished.set()
ctx.stop_event.set()
robot_voice.stop()
worker.join(timeout=5)
# perception and the mount camera first, so nothing is still publishing at the arm
for proc in (perception_proc, camera_proc):
    _stop_side(proc)
arm.on_cached_joints = None
observer.close()
console.close()
twin.close()  # the arm client, the scene, and the twin program
print("twin closed")
